<a href="https://colab.research.google.com/github/b0tt0mturn/WaveDash/blob/main/SurfAssessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Define project path inside your Drive
project_dir = '/content/drive/MyDrive/NOAA_Wave_Analysis'
os.makedirs(project_dir, exist_ok=True)

# Change the current working directory to your project folder
%cd {project_dir}

/content/drive/MyDrive/NOAA_Wave_Analysis


In [2]:
CACHE_DIR = os.path.join(project_dir, 'ndbc_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Data will be cached in: {CACHE_DIR}")

Data will be cached in: /content/drive/MyDrive/NOAA_Wave_Analysis/ndbc_cache


In [3]:
from logging import CRITICAL
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests # Added requests import
import seaborn as sns # Added seaborn for heatmap

# Wave Spectral Analysis Dashboard
# ============================================================
# SETTINGS
# ============================================================

# Define base URL for NDBC data
NDBC_BASE_URL = "https://www.ndbc.noaa.gov/data/realtime2/" # Base URL for data files

# Define the Station ID
STATION_ID = "41159"

# Construct URLs for the files
SPECTRUM_FILE = NDBC_BASE_URL + STATION_ID + ".data_spec"
ALPHA1_FILE   = NDBC_BASE_URL + STATION_ID + ".swdir"
ALPHA2_FILE   = NDBC_BASE_URL + STATION_ID + ".swdir2"

# Hours of prior data to plot
HOURS_TO_PLOT = 4

# Critical Values - 120 for Gunmounts
CRITICAL_ANGLE = 120
CRITICAL_PERIOD = 9

# Frequency range to search for meaningful wave systems
MIN_FREQ = 0.05
MAX_FREQ = 0.20

#Plot Limits
MAX_PERIOD_FOR_PLOTS = 18

# Number of bins on either side of a spectral peak
# ±2 = five bins total
BINS_AROUND_PEAK = 2

# Minimum spectral density required for a peak
PEAK_THRESHOLD = 0.1

# Minimum separation between detected peaks, in bins
MIN_PEAK_SEPARATION = 2

# Maximum number of wave systems to retain
MAX_PEAKS = 4

# New parameters for bubble sizing in polar plot
BUBBLE_SIZE_SCALING_METHOD = "radius" # Options: "area", "radius"
BUBBLE_SIZE_DETERMINANT = "peak_energy" # Options: "spectral_energy", "peak_energy"

# Heatmap resolution
BIN_RESOLUTION = 1 # seconds, for heatmap vertical resolution


# ============================================================
# HELPER FUNCTIONS
# ============================================================

pair_re = re.compile(
    r'([-+]?\d+(?:\.\d+)?)\s+\(([-+]?\d+(?:\.\d+)?)\)'
)

def read_ndbc_spectral_file(url, cache_dir=None):
    rows = []
    frequencies = None
    content_lines = None

    # Generate a cache filename from the URL, replacing non-alphanumeric chars
    cache_filename = re.sub(r'[^a-zA-Z0-9_.]', '_', url.replace(NDBC_BASE_URL, ''))
    cache_filepath = os.path.join(cache_dir, cache_filename) if cache_dir else None

    if cache_filepath and os.path.exists(cache_filepath):
        print(f"Loading from cache: {cache_filepath}")
        with open(cache_filepath, 'r') as f:
            content_lines = f.readlines()
    else:
        print(f"Downloading from: {url}")
        response = requests.get(url)
        response.raise_for_status()
        content_lines = response.text.splitlines()

        if cache_filepath:
            with open(cache_filepath, 'w') as f:
                f.write(response.text)
                print(f"Saved to cache: {cache_filepath}")

    if not content_lines:
        return pd.DataFrame(), None

    for line in content_lines:
            if line.startswith("#") or not line.strip():
                continue

            parts = line.split()

            if len(parts) < 6:
                continue

            pairs = pair_re.findall(line)

            if not pairs:
                continue

            if frequencies is None:
                frequencies = np.array([float(freq) for value, freq in pairs])

            timestamp = pd.Timestamp(
                int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
            )

            values = [float(value) for value, freq in pairs]

            rows.append((timestamp, values))

    dataframe = pd.DataFrame(
        [row[1] for row in rows],
        index=[row[0] for row in rows]
    )
    return dataframe, frequencies


def compass_direction(degrees):
    labels = ["N", "NNE", "NE", "ENE", "E", "ESE","SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
    index = int(((degrees + 11.25) % 360) // 22.5)  # this is the parameter that creates the proper index calculation
    return labels[index]


def analyze_spectral_data(timestamp, energy_df, alpha1_df, alpha2_df, frequencies, periods,
                          MIN_FREQ, MAX_FREQ, BINS_AROUND_PEAK, PEAK_THRESHOLD,
                          MIN_PEAK_SEPARATION, MAX_PEAKS):

    E = energy_df.loc[timestamp].to_numpy(dtype=float)
    D1 = alpha1_df.loc[timestamp].to_numpy(dtype=float)
    D2 = alpha2_df.loc[timestamp].to_numpy(dtype=float)

    search_mask = ((frequencies >= MIN_FREQ) & (frequencies <= MAX_FREQ))
    candidate_indices = np.where(search_mask)[0]

    candidates = []
    for idx in candidate_indices[1:-1]:
        if (E[idx] >= E[idx - 1] and E[idx] >= E[idx + 1] and E[idx] >= PEAK_THRESHOLD):
            candidates.append(idx)

    candidates = sorted(candidates, key=lambda i: E[i], reverse=True)

    peak_indices = []
    for idx in candidates:
        if all(abs(idx - j) >= MIN_PEAK_SEPARATION for j in peak_indices):
            peak_indices.append(idx)
        if len(peak_indices) >= MAX_PEAKS:
            break

    clusters = []
    for peak_index in peak_indices:
        start = max(0, peak_index - BINS_AROUND_PEAK)
        end = min(len(frequencies), peak_index + BINS_AROUND_PEAK + 1)

        freq = frequencies[start:end]
        period = periods[start:end]
        energy_values = E[start:end]
        dir1 = D1[start:end]
        dir2 = D2[start:end]

        combined_period = (np.sum(energy_values * period) / np.sum(energy_values)) if np.sum(energy_values) > 0 else 0

        directions_rad = np.deg2rad(np.concatenate([dir1, dir2]))
        weights = np.concatenate([energy_values, energy_values])
        if np.sum(weights) > 0:
            mean_direction = (np.rad2deg(np.arctan2(np.sum(weights * np.sin(directions_rad)), np.sum(weights * np.cos(directions_rad)))) + 360) % 360
        else:
            mean_direction = np.nan

        spectral_energy = np.sum(energy_values)

        clusters.append({
            "peak_index": peak_index,
            "peak_frequency": frequencies[peak_index],
            "peak_period": periods[peak_index],
            "combined_period": combined_period,
            "direction": mean_direction,
            "peak_energy": E[peak_index],
            "spectral_energy": spectral_energy,
            "frequency_min": freq.min(),
            "frequency_max": freq.max(),
            "period_min": period.min(),
            "period_max": period.max(),
            "start_index": start,
            "end_index": end
        })
    clusters = sorted(clusters, key=lambda x: x["spectral_energy"], reverse=True)
    return clusters


# ============================================================
# LOAD DATA
# ============================================================

energy, frequencies = read_ndbc_spectral_file(
    SPECTRUM_FILE, cache_dir=CACHE_DIR
)

alpha1, frequencies1 = read_ndbc_spectral_file(
    ALPHA1_FILE, cache_dir=CACHE_DIR
)

alpha2, frequencies2 = read_ndbc_spectral_file(
    ALPHA2_FILE, cache_dir=CACHE_DIR
)


# Make sure all three files use the same frequency bins
if not np.allclose(frequencies, frequencies1):
    raise ValueError("Frequency bins in alpha1 don't match spectrum.")

if not np.allclose(frequencies, frequencies2):
    raise ValueError("Frequency bins in alpha2 don't match spectrum.")


# Convert frequency to period
periods = 1.0 / frequencies


# ============================================================
# PREPARE DATA FOR HEATMAP
# ============================================================

# Find timestamps present in all three datasets
common_times = (
    energy.index
    .intersection(alpha1.index)
    .intersection(alpha2.index)
    .sort_values()
)

num_hours_to_plot = min(HOURS_TO_PLOT, len(common_times)) # Ensure we don't try to access more data than available

# Filter periods up to 18 seconds (inclusive) for the heatmap
MAX_PERIOD_FOR_HEATMAP = 20
filtered_periods_mask = (periods <= MAX_PERIOD_FOR_HEATMAP)
filtered_periods = periods[filtered_periods_mask]

# Using BIN_RESOLUTION-second bins for the heatmap
heatmap_period_breaks = np.arange(0, MAX_PERIOD_FOR_PLOTS + BIN_RESOLUTION, BIN_RESOLUTION)
heatmap_period_labels = np.arange(0, MAX_PERIOD_FOR_PLOTS, BIN_RESOLUTION)

heatmap_data_all_hours = []

# Iterate from the oldest of the last 'num_hours_to_plot' to the newest to prepare heatmap data
for k in range(num_hours_to_plot):
    current_ts = common_times[-(num_hours_to_plot - k)]
    current_hourly_index = k # This will range from 0 to num_hours_to_plot-1

    E_current = energy.loc[current_ts].to_numpy(dtype=float)
    E_filtered = E_current[filtered_periods_mask] # Apply the same period filter to energy

    # Create a temporary DataFrame for the current hour's spectral data
    current_hour_df = pd.DataFrame({
        'period': filtered_periods,
        'energy': E_filtered
    })

    # Bin periods and calculate max energy per bin for the current hour
    current_hour_df['binned_period'] = pd.cut(current_hour_df['period'],
                                                bins=heatmap_period_breaks,
                                                right=False,
                                                labels=heatmap_period_labels)
    current_hour_df['binned_period'] = pd.to_numeric(current_hour_df['binned_period'], errors='coerce')

    # Use fillna(0) to ensure bins with no data get 0 energy
    aggregated_energy_per_bin = current_hour_df.groupby('binned_period')['energy'].max().fillna(0)

    # Store this aggregated data for the current hour
    for binned_p, max_e in aggregated_energy_per_bin.items():
        if not pd.isna(binned_p):
            heatmap_data_all_hours.append({
                'hour': current_hourly_index,
                'binned_period': binned_p,
                'energy': max_e
            })

# Convert to DataFrame for easier pivoting
heatmap_df_full = pd.DataFrame(heatmap_data_all_hours)

# Pivot to create the heatmap matrix (periods as index, hours as columns)
# Reindex the rows to ensure all period bins from 0 to MAX_PERIOD_FOR_HEATMAP-1 are present
heatmap_matrix = heatmap_df_full.pivot_table(index='binned_period', columns='hour', values='energy', fill_value=0)
heatmap_matrix = heatmap_matrix.reindex(index=np.arange(0, MAX_PERIOD_FOR_PLOTS, BIN_RESOLUTION), fill_value=0).sort_index(ascending=False)

# Ensure all hours (columns) are present, filling with 0 if an hour had no data
all_hours_range = np.arange(num_hours_to_plot)
heatmap_matrix = heatmap_matrix.reindex(columns=all_hours_range, fill_value=0)

# Get global min/max for heatmap energy for color normalization across all plots
global_max_heatmap_energy = heatmap_matrix.values.max()
global_min_heatmap_energy = heatmap_matrix.values.min()


# ============================================================
# PRE-CALCULATE GLOBAL MAX ENERGY FOR STATIC BUBBLE SCALING
# ============================================================

global_max_spectral_energy = 0
global_max_peak_energy = 0

for k in range(num_hours_to_plot):
    current_ts_for_global_max = common_times[-(num_hours_to_plot - k)]
    clusters_for_global_max = analyze_spectral_data(current_ts_for_global_max, energy, alpha1, alpha2, frequencies, periods,
                                                     MIN_FREQ, MAX_FREQ, BINS_AROUND_PEAK, PEAK_THRESHOLD,
                                                     MIN_PEAK_SEPARATION, MAX_PEAKS)
    if clusters_for_global_max:
        current_max_spec_energy = max(c["spectral_energy"] for c in clusters_for_global_max)
        current_max_peak_energy = max(c["peak_energy"] for c in clusters_for_global_max)

        global_max_spectral_energy = max(global_max_spectral_energy, current_max_spec_energy)
        global_max_peak_energy = max(global_max_peak_energy, current_max_peak_energy)

# Ensure global maxes are not zero to avoid division by zero or overly large bubbles
if global_max_spectral_energy == 0:
    global_max_spectral_energy = 1
if global_max_peak_energy == 0:
    global_max_peak_energy = 1


# ============================================================
# PLOT GENERATION LOOP
# ============================================================

# Set default font size for all plots
plt.rcParams['font.size'] = 14

all_significant_wave_heights = [] # List to store significant wave heights

# Iterate over the last 24 common observations to generate plots
for i in range(1, num_hours_to_plot + 1):
    timestamp = common_times[-i]
    # hourly_index will range from num_hours_to_plot-1 (latest) down to 0 (oldest)
    hourly_index_for_marker = num_hours_to_plot - i

    print(f"\nAnalyzing: {timestamp}")

    # Calculate Significant Wave Height (Hs)
    E_current_for_Hs = energy.loc[timestamp].to_numpy(dtype=float)
    m0 = np.trapezoid(E_current_for_Hs, frequencies)
    Hs = 4 * np.sqrt(m0)
    all_significant_wave_heights.append({'timestamp': timestamp, 'significant_wave_height_meters': Hs})
    print(f"Significant Wave Height (Hs) at {timestamp}: {Hs * 3.281:.2f} feet")

    # Get clusters for the current timestamp using the refactored function
    clusters = analyze_spectral_data(timestamp, energy, alpha1, alpha2, frequencies, periods,
                                     MIN_FREQ, MAX_FREQ, BINS_AROUND_PEAK, PEAK_THRESHOLD,
                                     MIN_PEAK_SEPARATION, MAX_PEAKS)

    # ============================================================
    # PRINT RESULTS
    # ============================================================

    print(
        f"{'Peak':<8}"
        f"{'Period':>10}"
        f"{'Direction':>14}"
        f"{'Peak Energy':>14}"
        f"{'Cluster Energy':>17}"
    )

    print("-" * 65)


    for n, cluster in enumerate(clusters, 1):

        direction = compass_direction(
            cluster["direction"]
        )

        print(
            f"{n:<8}"
            f"{cluster['combined_period']:>8.2f} s"
            f"{cluster['direction']:>8.1f}\u00B0 {direction:>4}"
            f"{cluster['peak_energy']:>14.3f}"
            f"{cluster['spectral_energy']:>17.5f}"
        )

    # ============================================================
    # POLAR PLOT, SPECTRAL DENSITY PLOT, AND HEATMAP
    # ============================================================

    fig = plt.figure(figsize=(20, 15)) # Increased height for 3rd plot
    gs = fig.add_gridspec(2, 2) # 2 rows, 2 columns

    ax_polar = fig.add_subplot(gs[0, 0], projection='polar') # Top-left
    ax_spectral = fig.add_subplot(gs[0, 1]) # Top-right
    ax_heatmap = fig.add_subplot(gs[1, :]) # Bottom row, spans both columns


    # --- Polar Plot (ax_polar) ----
    ax_polar.set_theta_zero_location("N")
    ax_polar.set_theta_direction(-1)

    polar_legend_handles = []

    # Add Critical Angle line
    critical_angle_line, = ax_polar.plot([np.deg2rad(CRITICAL_ANGLE), np.deg2rad(CRITICAL_ANGLE)],
                  [0, MAX_PERIOD_FOR_PLOTS], color='blue', linestyle='--',
                  linewidth=2, label=f'Critical Angle ({CRITICAL_ANGLE}\u00B0)', zorder=0)
    polar_legend_handles.append(critical_angle_line)

    # Add Critical Period circle
    critical_period_circle, = ax_polar.plot(np.linspace(0, 2*np.pi, 100), [CRITICAL_PERIOD]*100,
                  color='blue', linestyle='--', linewidth=2,
                  label=f'Critical Period ({CRITICAL_PERIOD}s)', zorder=0)
    polar_legend_handles.append(critical_period_circle)

    for n, cluster in enumerate(clusters, 1):
        theta = np.deg2rad(cluster["direction"])
        radius = cluster["combined_period"]

        # Determine the value to use for sizing
        current_determinant_value = 0
        current_max_determinant_value = 1 # Avoid division by zero

        if BUBBLE_SIZE_DETERMINANT == "spectral_energy":
            current_determinant_value = cluster["spectral_energy"]
            current_max_determinant_value = global_max_spectral_energy # Use global max
        elif BUBBLE_SIZE_DETERMINANT == "peak_energy":
            current_determinant_value = cluster["peak_energy"]
            current_max_determinant_value = global_max_peak_energy # Use global max
        # Default to spectral energy if option is invalid
        else:
            current_determinant_value = cluster["spectral_energy"]
            current_max_determinant_value = global_max_spectral_energy # Use global max

        # Calculate the scaled factor
        scaled_factor = current_determinant_value / current_max_determinant_value if current_max_determinant_value > 0 else 0

        # Base size for scatter plot (s parameter controls area)
        base_scatter_area = 1000 # A visual scaling constant for default area size

        if BUBBLE_SIZE_SCALING_METHOD == "area":
            size = base_scatter_area * scaled_factor
        elif BUBBLE_SIZE_SCALING_METHOD == "radius":
            # If we want radius to scale linearly, the area (s) must scale quadratically
            # We use a visual base radius to keep sizes reasonable
            base_visual_radius = 70 # Adjust this for desired visual range of bubble sizes
            size = (base_visual_radius * scaled_factor)**2
        else: # Default to area scaling if method is invalid
            size = base_scatter_area * scaled_factor

        # Ensure a minimum size for visibility
        size = max(size, 30) # Reduced minimum size slightly for more visual range

        # Determine color based on heatmap intensity
        # Ensure binned_period_for_color is within the heatmap_matrix index range
        # It needs to be an actual index, so we round to nearest BIN_RESOLUTION and clamp
        binned_period_for_color = round(cluster['peak_period'] / BIN_RESOLUTION) * BIN_RESOLUTION
        binned_period_for_color = max(0, min(binned_period_for_color, MAX_PERIOD_FOR_HEATMAP - BIN_RESOLUTION))

        current_hour_col_index = hourly_index_for_marker

        # Get the corresponding energy value from the heatmap matrix
        # Use .get() with a default of 0 in case a bin/hour combination is missing
        energy_for_color_mapping = heatmap_matrix.loc[binned_period_for_color, current_hour_col_index]

        # Normalize the energy value for colormap
        if global_max_heatmap_energy > global_min_heatmap_energy:
            normalized_color_value = (energy_for_color_mapping - global_min_heatmap_energy) / (global_max_heatmap_energy - global_min_heatmap_energy)
        else:
            normalized_color_value = 0.5 # Default to middle color if no variation

        bubble_color = plt.cm.YlGnBu(normalized_color_value)

        ax_polar.scatter(theta, radius, s=size, alpha=0.70, edgecolors="black", linewidths=1.2, color=bubble_color)
        ax_polar.scatter(theta, radius, s=10, color='red', zorder=10) # Add a small red dot at the center

        label = (
            f"Swell {n}\n"
            #f"{cluster['combined_period']:.1f} s\n"
            #f"({cluster['direction']:.0f}\u00B0)\n"
        )

        ax_polar.annotate(label, (theta, radius), xytext=(8, 8), textcoords="offset points", ha="left", va="bottom")

    # --- Add highlighting for potential incoming swells (period > 13s, not in main clusters) ---
    E_current = energy.loc[timestamp].to_numpy(dtype=float)
    D1_current = alpha1.loc[timestamp].to_numpy(dtype=float)

    # Find all peaks above a certain low energy threshold within the relevant frequency range
    search_mask = ((frequencies >= MIN_FREQ) & (frequencies <= MAX_FREQ))
    candidate_indices_all = np.where(search_mask)[0]

    potential_swells_added_to_legend = False # Flag to add legend entry only once

    for peak_idx in candidate_indices_all[1:-1]: # Iterate through all potential peaks
        # Filter for local maxima with some energy (e.g., > 0.005, a small value)
        if (E_current[peak_idx] >= E_current[peak_idx - 1] and
            E_current[peak_idx] >= E_current[peak_idx + 1] and
            E_current[peak_idx] > 0.005): # Use a low energy threshold to catch faint peaks

            current_period = periods[peak_idx]

            # Check if period is greater than 13 seconds
            if current_period > 13:
                # Check if this peak is already part of an identified cluster
                is_already_clustered = False
                for cluster in clusters:
                    if cluster["start_index"] <= peak_idx <= cluster["end_index"]:
                        is_already_clustered = True
                        break

                if not is_already_clustered:
                    # Calculate direction for this individual peak
                    direction_deg = D1_current[peak_idx] # Use D1 for direction
                    theta_alert = np.deg2rad(direction_deg)
                    radius_alert = current_period

                    ax_polar.scatter(theta_alert, radius_alert, s=50, color='lime', edgecolors='black', linewidths=1.5, zorder=10) # Plot green circle
                    if not potential_swells_added_to_legend:
                        # Create a dummy plot for the legend entry
                        dummy_handle, = ax_polar.plot([], [], 'o', color='lime', markeredgecolor='black', markeredgewidth=1.5, markersize=8, label='Potential Incoming Swell (>13s)')
                        polar_legend_handles.append(dummy_handle)
                        potential_swells_added_to_legend = True

    # ax_polar.legend(handles=polar_legend_handles, loc='best', bbox_to_anchor=(1.1, 1.1), fontsize='small', title="Key")

    ax_polar.set_rlim(0, MAX_PERIOD_FOR_PLOTS)
    ax_polar.set_rgrids([4, 6, 8, 10, 12, 14, 16], labels=[]) # Set radial grid lines and suppress labels
    # Manually add period labels at 315 degrees
    for p in [4, 6, 8, 10, 12, 14, 16]:
        ax_polar.text(np.deg2rad(315), p, f'{p} s', ha='center', va='center', color='black')

    ax_polar.set_title(
        "Buoy " + STATION_ID + " Spectral Peak-Cluster Summary\n"
        f"{timestamp:%Y-%m-%d %H:%M} UTC\n"
        "Each bubble = spectral peak \u00B1 "
        f"{BINS_AROUND_PEAK} bins",
        pad=25
    )
    ax_polar.grid(True, alpha=0.3)


    # --- Spectral Energy Density Plot (ax_spectral) ---
    E_for_plot = energy.loc[timestamp].to_numpy(dtype=float)
    line_plot, = ax_spectral.plot(periods, E_for_plot, label='Spectral Energy Density', color='blue')
    ax_spectral.set_xlabel('Wave Period (s)')
    ax_spectral.set_ylabel('Spectral Energy Density')
    ax_spectral.set_title(f'Spectral Energy Density vs. Wave Period\n{timestamp:%Y-%m-%d %H:%M} UTC Significant Wave Height: {Hs * 3.281:.1f} feet')
    ax_spectral.grid(True, linestyle='--', alpha=0.7)
    ax_spectral.set_xlim(right=MAX_PERIOD_FOR_PLOTS)
    ax_spectral.set_ylim(bottom=0, top=E_for_plot.max() * 1.1 if E_for_plot.max() > 2.5 else 2.5) # Dynamic y-limit

    legend_entries = [line_plot]
    for n, cluster in enumerate(clusters, 1):
        ax_spectral.axvline(x=cluster['peak_period'], color='r', linestyle='--', alpha=0.7, zorder=0)
        ax_spectral.text(cluster['peak_period'], ax_spectral.get_ylim()[1]*0.9, f"  {cluster['peak_period']:.1f} s", color='r', rotation=90, va='top')

        direction = compass_direction(cluster["direction"])
        cluster_legend_label = (
            f" {n}: {cluster['combined_period']:.1f} s, "
            f"{cluster['direction']:.0f}\u00B0, "
            f"P.E.: {cluster['peak_energy']:.2f}, "
            #f"Significant Wave Height: {Hs * 3.281:.2f} feet"
        )
        dummy_handle, = ax_spectral.plot([], [], 'o', color=plt.cm.viridis(n / (len(clusters) + 1)), label=cluster_legend_label)
        legend_entries.append(dummy_handle)

    ax_spectral.legend(handles=legend_entries, loc='upper left', title="Peak Details")


    # --- Heatmap (ax_heatmap) ---
    sns.heatmap(heatmap_matrix, ax=ax_heatmap, cmap='jet', cbar_kws={'label': 'Spectral Energy Density'}, linewidths=0.5, linecolor='lightgrey')
    ax_heatmap.set_title(f'Full Spectral Energy Density vs. Period over Last {num_hours_to_plot} Hours\n{timestamp:%Y-%m-%d %H:%M} UTC')
    ax_heatmap.set_xlabel(f'Hour Index (0 = Oldest, {num_hours_to_plot-1} = Latest)')
    ax_heatmap.set_ylabel('Wave Period (s)')

    # Modified tick labels to show fewer ticks on the x-axis
    tick_interval = 4 # Display a tick every 12 hours
    x_ticks = np.arange(0, num_hours_to_plot, tick_interval) + 0.5 # Position ticks at the center of bins
    x_labels = np.arange(0, num_hours_to_plot, tick_interval)

    ax_heatmap.set_xticks(x_ticks)
    ax_heatmap.set_xticklabels(x_labels, rotation=0)

    # Add vertical marker for the current hour
    ax_heatmap.axvline(x=hourly_index_for_marker+0.5, color='blue', linestyle='--', linewidth=2, label=f'Current Hour: {hourly_index_for_marker}')
    ax_heatmap.legend(handles=[plt.Line2D([0], [0], color='blue', linestyle='--', linewidth=2)], labels=['Current Hour'], loc='lower right') # Add legend for the marker


    plt.tight_layout()

    # Save the figure with a consistent filename for overwriting
    # hourly_index_for_marker will create filenames from 0 (oldest) to num_hours_to_plot-1 (latest)
    filename = os.path.join(project_dir, f'spectral_analysis_plot_hour_{hourly_index_for_marker:02d}.png')
    fig.savefig(filename)
    plt.close(fig) # Close the figure to free up memory

print("\nGenerated and overwritten plots for the last 24 hours (or available data).")


Saved to cache: /content/drive/MyDrive/NOAA_Wave_Analysis/ndbc_cache/41159.data_spec
Saved to cache: /content/drive/MyDrive/NOAA_Wave_Analysis/ndbc_cache/41159.swdir
Saved to cache: /content/drive/MyDrive/NOAA_Wave_Analysis/ndbc_cache/41159.swdir2

Analyzing: 2026-09-02 21:00:00
Significant Wave Height (Hs) at 2026-09-02 21:00:00: 3.06 feet
Peak        Period     Direction   Peak Energy   Cluster Energy
-----------------------------------------------------------------
1          10.32 s   118.7°  ESE         1.670          4.99300
2           9.50 s   122.1°  ESE         0.985          4.69200

Analyzing: 2026-09-02 20:00:00
Significant Wave Height (Hs) at 2026-09-02 20:00:00: 3.06 feet
Peak        Period     Direction   Peak Energy   Cluster Energy
-----------------------------------------------------------------
1           9.98 s   118.8°  ESE         1.560          5.16500

Analyzing: 2026-09-02 19:00:00
Significant Wave Height (Hs) at 2026-09-02 19:00:00: 3.23 feet
Peak        Per

### Interactive Player for Wave Spectral Analysis Plots

Use the controls below to play, pause, and navigate through the hourly wave spectral analysis plots to observe trends in swell energy and direction with time.

In [4]:
from IPython.display import display, Image
import ipywidgets as widgets
from IPython.display import clear_output
import glob
import time
import threading # Import threading for non-blocking animation

# Get all generated PNG files, sorted by name (which includes timestamp)
image_files = sorted(glob.glob(os.path.join(project_dir, 'spectral_analysis_plot_hour_*.png')))

# UI for Wave Analysis
if not image_files:
    print("No image files found. Please ensure the previous cell ran successfully and generated the plots.")
else:
    # Display initial image
    current_image_index = 0
    image_widget = widgets.Image(
        value=open(image_files[current_image_index], 'rb').read(),
        format='png',
        width=900,
        height=450,
    )

    # Output widget to hold the image, so we can clear and update it
    image_output = widgets.Output()

    with image_output:
        display(image_widget)

    # Slider for manual navigation
    image_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(image_files) - 1,
        step=1,
        description='Image Index:',
        orientation='horizontal',
        continuous_update=False
    )

    def update_image(index):
        if 0 <= index < len(image_files):
            new_image_data = open(image_files[index], 'rb').read()
            with image_output:
                clear_output(wait=True) # Clear previous output in this output widget
                image_widget.value = new_image_data # Update the image widget's value
                display(image_widget) # Re-display the updated image widget

    # Link slider to image update function
    def on_slider_change(change):
        global current_image_index
        current_image_index = change.new
        update_image(current_image_index)

    image_slider.observe(on_slider_change, names='value')

    # Control buttons
    play_button = widgets.Button(description="Play")
    pause_button = widgets.Button(description="Pause")
    next_button = widgets.Button(description="Next")
    prev_button = widgets.Button(description="Previous")
    first_button = widgets.Button(description="First") # New button
    last_button = widgets.Button(description="Last")   # New button

    # Playback control variables
    playing = False
    animation_thread = None # To hold the animation thread

    def animation_loop():
        global playing, current_image_index
        while playing and current_image_index < len(image_files) - 1:
            current_image_index += 1
            image_slider.value = current_image_index # This triggers update_image via observer
            time.sleep(1.0) # Adjust speed here (seconds per frame)
        playing = False # Animation finished or stopped

    def on_play_button_click(b):
        global playing, animation_thread
        if not playing:
            playing = True
            # Ensure no old thread is lingering and if so, stop it
            if animation_thread is not None and animation_thread.is_alive():
                playing = False
                animation_thread.join(timeout=1.5) # Wait for it to finish gracefully

            animation_thread = threading.Thread(target=animation_loop)
            animation_thread.start()

    def on_pause_button_click(b):
        global playing
        playing = False # This will stop the animation_loop

    def on_next_button_click(b):
        global current_image_index
        on_pause_button_click(None) # Pause animation on manual navigation
        if current_image_index < len(image_files) - 1:
            current_image_index += 1
            image_slider.value = current_image_index

    def on_prev_button_click(b):
        global current_image_index
        on_pause_button_click(None) # Pause animation on manual navigation
        if current_image_index > 0:
            current_image_index -= 1
            image_slider.value = current_image_index

    def on_first_button_click(b):
        global current_image_index
        on_pause_button_click(None) # Pause animation on manual navigation
        current_image_index = 0
        image_slider.value = current_image_index

    def on_last_button_click(b):
        global current_image_index
        on_pause_button_click(None) # Pause animation on manual navigation
        current_image_image_index = len(image_files) - 1
        image_slider.value = current_image_index

    play_button.on_click(on_play_button_click)
    pause_button.on_click(on_pause_button_click)
    next_button.on_click(on_next_button_click)
    prev_button.on_click(on_prev_button_click)
    first_button.on_click(on_first_button_click)
    last_button.on_click(on_last_button_click)

    controls = widgets.HBox([first_button, prev_button, play_button, pause_button, next_button, last_button])
    display(controls, image_slider, image_output)


IntSlider(value=0, continuous_update=False, description='Image Index:', max=23)

Output()

### Generate Animated GIF and MP4 Video

This section will create an animated GIF and an MP4 video from the individual hourly plots. These files will be saved in your project directory (`/content/drive/MyDrive/NOAA_Wave_Analysis`).

In [ ]:
# Install necessary libraries for GIF and MP4 creation
%pip install imageio imageio-ffmpeg

In [5]:
import imageio
import os
import glob

# Get all generated PNG files, sorted by name (which includes timestamp)
image_files = sorted(glob.glob(os.path.join(project_dir, 'spectral_analysis_plot_hour_*.png')))

if not image_files:
    print("No image files found for creating GIF/MP4.")
else:
    # Read all images into a list
    images = []
    for filename in image_files:
        images.append(imageio.imread(filename))

    # Define output filenames
    gif_filename = os.path.join(project_dir, 'spectral_analysis_animation.gif')
    # mp4_filename = os.path.join(project_dir, 'spectral_analysis_video.mp4')

    # Generate GIF (1 frame per second)
    imageio.mimsave(gif_filename, images, fps=1)
    print(f"Generated GIF: {gif_filename}")

    # Generate MP4 (1 frame per second)
    # Ensure 'imageio-ffmpeg' is installed for mp4 creation
    # imageio.mimsave(mp4_filename, images, fps=1, codec='libx264', quality=8) # quality 8 is good for general use
    # print(f"Generated MP4: {mp4_filename}")

    print("Animation generation complete.")


/tmp/ipykernel_4947/1930855159.py:14: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(filename))


Generated GIF: /content/drive/MyDrive/NOAA_Wave_Analysis/spectral_analysis_animation.gif
Animation generation complete.


In [ ]:
fig.savefig(os.path.join(project_dir, 'spectral_analysis_plot.png'))